# 🎓 수학교육평가론 — 협력학습 MVP (v2)

## 아키텍처
```
User ──▶ LLM(학습자 분석) ──▶ Learner Model ──▶ Tutor Model ◀── Resource
                                                    │
                                                    ▼ (지시)
                                 Persona ─▶ LLM ─▶ AI Student 1, 2 ─▶ User
```

## Task
**소수와 합성수의 개념 탐구** (Stage 1~4)

## 외부 파일 구조
- `config/personas.json` — AI 학생 2명 페르소나
- `config/learner_model.json` — 학습자 모델 스키마
- `config/tutor_model.json` — 교수자 모델 정책
- `config/tasks.json` — Stage 1~4 과제 정의
- `prompts/*.md` — 각 단계별 프롬프트 템플릿


In [ ]:
# 셀 1: 설치 및 API 키
!pip install anthropic -q

import anthropic, json, os, re
from pathlib import Path
from google.colab import userdata, files

client = anthropic.Anthropic(api_key=userdata.get('CLAUDE_API_KEY'))
MODEL = "claude-sonnet-4-20250514"

print("✅ Anthropic 클라이언트 준비")


## 📁 config / prompts 폴더 업로드

아래 셀을 실행한 뒤, `config/` 폴더와 `prompts/` 폴더의 파일들을 모두 업로드하세요.
Colab에서는 폴더 업로드가 제한적이므로 **파일을 한번에 다중 선택**하여 업로드합니다.
업로드 후 자동으로 폴더로 분류됩니다.


In [ ]:
# 셀 2: config/prompts 파일 업로드 (처음 1회)
# 이미 드라이브에 마운트되어 있다면 이 셀은 건너뛰어도 됨.

CONFIG_DIR = Path("config")
PROMPTS_DIR = Path("prompts")
CONFIG_DIR.mkdir(exist_ok=True)
PROMPTS_DIR.mkdir(exist_ok=True)

print("config/ 및 prompts/ 폴더의 모든 파일을 선택해서 업로드하세요.")
uploaded = files.upload()

for fname in uploaded:
    src = Path(fname)
    if fname.endswith(".json"):
        src.rename(CONFIG_DIR / fname)
    elif fname.endswith(".md"):
        src.rename(PROMPTS_DIR / fname)

print(f"✅ config: {sorted(os.listdir(CONFIG_DIR))}")
print(f"✅ prompts: {sorted(os.listdir(PROMPTS_DIR))}")


In [ ]:
# 셀 3: 설정 파일 로드

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_md(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

CONFIG = {
    "personas": load_json("config/personas.json"),
    "learner_model_schema": load_json("config/learner_model.json"),
    "tutor_model": load_json("config/tutor_model.json"),
    "tasks": load_json("config/tasks.json"),
}

PROMPTS = {
    "learner_analysis": load_md("prompts/01_learner_analysis.md"),
    "tutor_decision": load_md("prompts/02_tutor_decision.md"),
    "ai_student": load_md("prompts/03_ai_student_utterance.md"),
    "stage_intro": load_md("prompts/04_stage_intro.md"),
    "stage_closure": load_md("prompts/05_stage_closure.md"),
    "misconception": load_md("prompts/06_misconception_challenge.md"),
    "encouragement": load_md("prompts/07_encouragement.md"),
}

print(f"✅ Task: {CONFIG['tasks']['task_title']}")
print(f"✅ Stages: {len(CONFIG['tasks']['stages'])}")
print(f"✅ AI 학생: {[p['name'] for p in CONFIG['personas']['ai_students'].values()]}")
print(f"✅ 학습자 모델 영역: {list(CONFIG['learner_model_schema']['models'].keys())}")
print(f"✅ 프롬프트 파일: {list(PROMPTS.keys())}")


In [ ]:
# 셀 4: 학습자 모델 인스턴스 초기화 (사용자 + AI 2명)

def create_learner_model_instance(student_name, override_initial=None):
    schema = CONFIG["learner_model_schema"]["models"]
    inst = {"student_name": student_name, "models": {}}
    for mk, mv in schema.items():
        inst["models"][mk] = {}
        for dk, dv in mv["dimensions"].items():
            init_val = dv.get("default", 3)
            if override_initial and mk in override_initial and dk in override_initial[mk]:
                init_val = override_initial[mk][dk]
            inst["models"][mk][dk] = {
                "value": init_val,
                "history": [{"stage": 0, "value": init_val, "evidence": "초기값"}]
            }
    return inst

learner_models = {
    "user": create_learner_model_instance("사용자"),
    "ai_1": create_learner_model_instance(
        CONFIG["personas"]["ai_students"]["ai_1"]["name"],
        CONFIG["personas"]["ai_students"]["ai_1"]["initial_learner_state"]
    ),
    "ai_2": create_learner_model_instance(
        CONFIG["personas"]["ai_students"]["ai_2"]["name"],
        CONFIG["personas"]["ai_students"]["ai_2"]["initial_learner_state"]
    )
}

print("✅ 학습자 모델 인스턴스 3개 초기화 완료")
for k, v in learner_models.items():
    cs = v["models"]["cognitive_state"]["conceptual_understanding"]["value"]
    print(f"  {v['student_name']}: 개념이해도={cs}/5")


In [ ]:
# 셀 5: Claude API 호출 헬퍼

def call_claude(prompt, max_tokens=1000, temperature=0.7):
    """단일 메시지로 Claude 호출."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text

def extract_json(text):
    """LLM 응답에서 JSON 블록 추출."""
    # ```json ... ``` 블록 우선
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        return json.loads(m.group(1))
    # 중괄호로 시작하는 첫 JSON 객체
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        return json.loads(text[start:end])
    raise ValueError("JSON을 추출할 수 없습니다:\n" + text)

def render_prompt(template, vars_dict):
    """{{var}} 치환."""
    out = template
    for k, v in vars_dict.items():
        placeholder = "{{" + k + "}}"
        out = out.replace(placeholder, str(v) if not isinstance(v, (dict, list)) else json.dumps(v, ensure_ascii=False, indent=2))
    return out

print("✅ API 헬퍼 준비")


In [ ]:
# 셀 6: 협력학습 세션 (3-stage 파이프라인 구현)

class CollaborativeSession:
    def __init__(self):
        self.task = CONFIG["tasks"]
        self.current_stage = 1
        self.conversation = []   # [{speaker, content, stage}]
        self.turn_count = 0
        self.stage_turn_count = 0
        self.last_tutor_decision = None

    # ----- 유틸 -----
    def recent_dialogue(self, n=6):
        recent = self.conversation[-n:]
        return "\n".join(f"[{m['speaker']}]: {m['content']}" for m in recent) or "(아직 대화 없음)"

    def current_stage_info(self):
        return self.task["stages"][str(self.current_stage)]

    def _flatten_lm(self, lm):
        """시각화·프롬프트용으로 학습자 모델 값만 평탄화."""
        return {
            mk: {dk: dv["value"] for dk, dv in mv.items()}
            for mk, mv in lm["models"].items()
        }

    # ----- 1단계: 학습자 분석 LLM -----
    def analyze_user_utterance(self, user_utterance):
        stage = self.current_stage_info()
        prompt = render_prompt(PROMPTS["learner_analysis"], {
            "task_title": self.task["task_title"],
            "stage_title": stage["title"],
            "core_question": stage["core_question"],
            "user_utterance": user_utterance,
            "recent_dialogue": self.recent_dialogue(6),
            "current_learner_model": self._flatten_lm(learner_models["user"]),
            "learner_model_schema": CONFIG["learner_model_schema"]["models"],
        })
        try:
            raw = call_claude(prompt, max_tokens=1200, temperature=0.3)
            result = extract_json(raw)
        except Exception as e:
            print(f"  ⚠️ 학습자 분석 실패: {e}")
            return {"updates": [], "misconception_changes": {"added": [], "removed": []}, "observation_summary": ""}

        # 학습자 모델 업데이트 (사용자만)
        for u in result.get("updates", []):
            mk, dk, nv = u["model"], u["dimension"], u["new_value"]
            if mk in learner_models["user"]["models"] and dk in learner_models["user"]["models"][mk]:
                learner_models["user"]["models"][mk][dk]["value"] = nv
                learner_models["user"]["models"][mk][dk]["history"].append({
                    "stage": self.current_stage, "value": nv, "evidence": u.get("evidence", "")
                })
        # 오개념 리스트
        if "misconceptions" in learner_models["user"]["models"]["cognitive_state"]:
            mis = learner_models["user"]["models"]["cognitive_state"]["misconceptions"]
            if not isinstance(mis["value"], list):
                mis["value"] = []
            for add in result.get("misconception_changes", {}).get("added", []):
                if add not in mis["value"]:
                    mis["value"].append(add)
            for rem in result.get("misconception_changes", {}).get("removed", []):
                if rem in mis["value"]:
                    mis["value"].remove(rem)
        return result

    # ----- 2단계: 교수자 모델 의사결정 -----
    def tutor_decision(self):
        stage = self.current_stage_info()
        prompt = render_prompt(PROMPTS["tutor_decision"], {
            "learning_objectives": self.task["learning_objectives"],
            "current_stage_full": stage,
            "user_learner_model": self._flatten_lm(learner_models["user"]),
            "ai_1_learner_model": self._flatten_lm(learner_models["ai_1"]),
            "ai_2_learner_model": self._flatten_lm(learner_models["ai_2"]),
            "recent_dialogue": self.recent_dialogue(6),
            "tutor_principles": CONFIG["tutor_model"]["tutor_model"]["pedagogical_principles"],
            "role_pool": CONFIG["tutor_model"]["tutor_model"]["ai_student_role_assignment"]["role_pool"],
        })
        try:
            raw = call_claude(prompt, max_tokens=1200, temperature=0.5)
            decision = extract_json(raw)
        except Exception as e:
            print(f"  ⚠️ 교수자 의사결정 실패: {e}")
            decision = {
                "strategy": "기본 진행",
                "ai_1_directive": {"role": "탐험가", "speech_goal": "자연스럽게 대화 진행", "must_include": "", "must_avoid": "정답 직접 말하기"},
                "ai_2_directive": {"role": "검증자", "speech_goal": "민준의 의견 검토", "must_include": "", "must_avoid": "정답 직접 말하기"},
                "pedagogical_goal": "",
                "stage_complete": False,
            }
        self.last_tutor_decision = decision
        return decision

    # ----- 3단계: AI 학생 발화 생성 -----
    def generate_ai_utterance(self, student_key, directive, user_utterance):
        persona = CONFIG["personas"]["ai_students"][student_key]
        stage = self.current_stage_info()
        prompt = render_prompt(PROMPTS["ai_student"], {
            "student_name": persona["name"],
            "my_persona": persona,
            "my_learner_state": self._flatten_lm(learner_models[student_key]),
            "stage_title": stage["title"],
            "stage_prompt": stage["prompt"],
            "role": directive.get("role", ""),
            "speech_goal": directive.get("speech_goal", ""),
            "must_include": directive.get("must_include", ""),
            "must_avoid": directive.get("must_avoid", ""),
            "recent_dialogue": self.recent_dialogue(8),
            "user_utterance": user_utterance,
        })
        return call_claude(prompt, max_tokens=400, temperature=0.9).strip()

    # ----- 전체 턴 -----
    def user_turn(self, user_utterance):
        self.conversation.append({"speaker": "사용자", "content": user_utterance, "stage": self.current_stage})
        self.turn_count += 1
        self.stage_turn_count += 1

        # 1) 학습자 분석
        print("  [1/3] 🔍 학습자 분석 중...")
        analysis = self.analyze_user_utterance(user_utterance)
        if analysis.get("observation_summary"):
            print(f"       · 관찰: {analysis['observation_summary']}")

        # 2) 교수자 의사결정
        print("  [2/3] 🎓 교수자 모델 의사결정 중...")
        decision = self.tutor_decision()
        print(f"       · 전략: {decision.get('strategy', '')}")

        # 3) AI 학생 발화
        print("  [3/3] 💬 AI 학생 발화 생성 중...")
        ai1_text = self.generate_ai_utterance("ai_1", decision["ai_1_directive"], user_utterance)
        self.conversation.append({"speaker": CONFIG["personas"]["ai_students"]["ai_1"]["name"], "content": ai1_text, "stage": self.current_stage})

        ai2_text = self.generate_ai_utterance("ai_2", decision["ai_2_directive"], user_utterance)
        self.conversation.append({"speaker": CONFIG["personas"]["ai_students"]["ai_2"]["name"], "content": ai2_text, "stage": self.current_stage})

        return {
            "analysis": analysis,
            "decision": decision,
            "ai_1": ai1_text,
            "ai_2": ai2_text,
        }

    # ----- Stage 전환 -----
    def advance_stage(self):
        if self.current_stage < 4:
            self.current_stage += 1
            self.stage_turn_count = 0
            return True
        return False

    def stage_intro_utterance(self, opener_key="ai_1"):
        persona = CONFIG["personas"]["ai_students"][opener_key]
        stage = self.current_stage_info()
        prompt = render_prompt(PROMPTS["stage_intro"], {
            "opener_name": persona["name"],
            "stage_title": stage["title"],
            "stage_prompt": stage["prompt"],
            "core_question": stage["core_question"],
            "my_persona": persona,
        })
        text = call_claude(prompt, max_tokens=300, temperature=0.8).strip()
        self.conversation.append({"speaker": persona["name"], "content": text, "stage": self.current_stage})
        return text

print("✅ CollaborativeSession 클래스 준비")


In [ ]:
# 셀 7: 한글 폰트 + 시각화

!apt-get -qq install fonts-nanum
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm, numpy as np
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False


def print_user_model():
    m = learner_models["user"]
    print(f"\n{'═'*55}\n📊 {m['student_name']}의 학습자 모델\n{'═'*55}")
    for mk, mv in CONFIG["learner_model_schema"]["models"].items():
        print(f"\n  ▸ {mv['name']}")
        for dk, dv in mv["dimensions"].items():
            val = m["models"][mk][dk]["value"]
            if isinstance(val, (int, float)):
                bar = "█"*int(val) + "░"*(5-int(val))
                print(f"    [{bar}] {dv['name']}: {val}/5")
            elif isinstance(val, list):
                if val:
                    print(f"    {dv['name']}: {val}")


def plot_radar_all():
    labels, vals = [], {"user": [], "ai_1": [], "ai_2": []}
    for mk, mv in CONFIG["learner_model_schema"]["models"].items():
        for dk, dv in mv["dimensions"].items():
            sample = learner_models["user"]["models"][mk][dk]["value"]
            if isinstance(sample, (int, float)):
                labels.append(dv["name"])
                for sk in ["user", "ai_1", "ai_2"]:
                    vals[sk].append(learner_models[sk]["models"][mk][dk]["value"])
    N = len(labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    colors = {"user": "#E74C3C", "ai_1": "#3498DB", "ai_2": "#2ECC71"}
    names = {"user": "사용자", "ai_1": CONFIG["personas"]["ai_students"]["ai_1"]["name"],
             "ai_2": CONFIG["personas"]["ai_students"]["ai_2"]["name"]}
    for sk in ["user", "ai_1", "ai_2"]:
        v = vals[sk] + vals[sk][:1]
        ax.plot(angles, v, 'o-', linewidth=2, label=names[sk], color=colors[sk])
        ax.fill(angles, v, alpha=0.15, color=colors[sk])
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, size=9)
    ax.set_ylim(0, 5); ax.set_yticks([1,2,3,4,5])
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
    ax.set_title("학습자 모델 비교 (Radar)", size=14, pad=20, fontweight='bold')
    plt.tight_layout(); plt.show()


def plot_user_history():
    """사용자의 인지·의사소통·감정·협력 변화 추이."""
    schema = CONFIG["learner_model_schema"]["models"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.flatten()
    for ax, (mk, mv) in zip(axes, schema.items()):
        for dk, dv in mv["dimensions"].items():
            hist = learner_models["user"]["models"][mk][dk]["history"]
            if isinstance(hist[0]["value"], (int, float)):
                xs = list(range(len(hist)))
                ys = [h["value"] for h in hist]
                ax.plot(xs, ys, 'o-', label=dv["name"], linewidth=2, markersize=6)
        ax.set_title(mv["name"], fontsize=12, fontweight='bold')
        ax.set_ylim(0, 5.5); ax.set_xlabel("업데이트 회차"); ax.set_ylabel("수준")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.suptitle("사용자 학습자 모델 — 단계별 변화", fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

print("✅ 시각화 함수 준비")


## 🚀 대화형 세션 실행

- 입력: 사용자 발화 (텍스트)
- 명령어:
  - `/radar` — 3명 레이더 차트
  - `/history` — 사용자 변화 추이
  - `/model` — 사용자 모델 상세
  - `/next` — 다음 stage로 수동 이동
  - `/decision` — 직전 교수자 의사결정 JSON 출력
  - `/quit` — 종료


In [ ]:
# 셀 8: 세션 실행

def run_session():
    session = CollaborativeSession()
    print(f"\n{'═'*60}\n🎓 {session.task['task_title']}\n{'═'*60}")

    # Stage 1 intro
    stage = session.current_stage_info()
    print(f"\n📝 {stage['title']}")
    print(f"   {stage['prompt']}\n")
    print("   (AI 학생이 먼저 말문을 엽니다...)\n")
    intro = session.stage_intro_utterance("ai_1")
    print(f"🧑‍🎓 {CONFIG['personas']['ai_students']['ai_1']['name']}: {intro}")

    while True:
        user_input = input("\n🙋 나: ").strip()
        if not user_input:
            continue
        if user_input.startswith("/"):
            cmd = user_input.lower()
            if cmd == "/quit":
                print("\n📊 최종 학습자 모델:")
                print_user_model(); plot_radar_all(); plot_user_history()
                break
            elif cmd == "/radar":
                plot_radar_all(); continue
            elif cmd == "/history":
                plot_user_history(); continue
            elif cmd == "/model":
                print_user_model(); continue
            elif cmd == "/next":
                if session.advance_stage():
                    print(f"\n▶ Stage {session.current_stage}로 이동")
                    stage = session.current_stage_info()
                    print(f"📝 {stage['title']}\n   {stage['prompt']}\n")
                    intro = session.stage_intro_utterance("ai_2")
                    print(f"🧑‍🎓 {CONFIG['personas']['ai_students']['ai_2']['name']}: {intro}")
                else:
                    print("  모든 stage 완료")
                continue
            elif cmd == "/decision":
                print(json.dumps(session.last_tutor_decision, ensure_ascii=False, indent=2))
                continue
            else:
                print("  알 수 없는 명령어"); continue

        result = session.user_turn(user_input)
        n1 = CONFIG['personas']['ai_students']['ai_1']['name']
        n2 = CONFIG['personas']['ai_students']['ai_2']['name']
        print(f"\n🧑‍🎓 {n1}: {result['ai_1']}")
        print(f"🧑‍🎓 {n2}: {result['ai_2']}")

        # stage 자동 전환
        if result["decision"].get("stage_complete"):
            print(f"\n✅ Stage {session.current_stage} 완료 신호")
            if session.advance_stage():
                stage = session.current_stage_info()
                print(f"\n▶ Stage {session.current_stage} 시작")
                print(f"📝 {stage['title']}\n   {stage['prompt']}\n")
                intro = session.stage_intro_utterance("ai_2")
                print(f"🧑‍🎓 {CONFIG['personas']['ai_students']['ai_2']['name']}: {intro}")
            else:
                print("\n🎉 모든 Stage 완료! /quit 으로 종료하세요.")

    return session

# 실행
session = run_session()
